In [1]:
import sys, pathlib
# Make the repo root importable regardless of where Jupyter launched from.
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(pathlib.Path.cwd().parent / '.env')

from fortyguard import FortyGuardClient
client = FortyGuardClient()
print('Base URL :', client.base_url)
print('API key  :', client.api_key[:6] + '…' if client.api_key else '(missing)')

Base URL : https://api.fortyguard.com
API key  : 618043…


In [ ]:
# Hit the credits endpoint as a cheap auth check.
from datetime import date, timedelta

end_date   = date.today().isoformat()
start_date = (date.today() - timedelta(days=30)).isoformat()

usage = client.fetch_api_key_custom_usage(start_date=start_date, end_date=end_date)

date_range = usage.get('date_range', {})
print(f"Window      : {date_range.get('date_range_formatted') or f'{start_date} → {end_date}'}")
print(f"Credits used: {usage.get('total_credits_used')}")

for row in usage.get('activity_breakdown', []):
    print(f"  {row.get('name'):>28}: {row.get('credits')} credits over {row.get('count')} calls")

Window      : Jul 25, 2026 – Aug 24, 2026
Credits used: 163260
            Heatmap Generation: 160360 credits over 38 calls
  Environment Parameter Analysis: 2900 credits over 1 calls


In [ ]:
benefits = [4, 8, 5, 10, 6]
max_stations = 5
best_cost = {}

for i in range(1, len(benefits)):
    best_cost[i] = {}
    for j in range(1, max_stations):
        if (j > i) : continue
        best_cost[i][j] = float('-inf')

best_cost[1][1] = benefits[0]

for i in range(1, len(benefits)):
    for j in range(1, max_stations):
        if (j > i) : continue

        print(f"----------------------Best Cost[{i}][{j}]----------------------")

        if (j == 1):
            best_cost[i][j] = max(benefits[0:i])

        else:
            if (i == j):
                best_cost[i][j] = sum(benefits[0:j])

            else:
                current_benefit = benefits[i - 1]
                print(f"Best Cost[{i - 1}][{j - 1}] = {best_cost[i - 1 ][j - 1]}")
                print(f"Best Cost[{i - 1}][{j}] = {best_cost[i - 1][j]}")
                print(f"Current Benefit : {current_benefit}")
                best_cost[i][j] = max((current_benefit + best_cost[i - 1][j - 1]), best_cost[i - 1][j])

        print(f"Best Cost[{i}][{j}] = {best_cost[i][j]}")

----------------------Best Cost[1][1]----------------------
Best Cost[1][1] = 1
----------------------Best Cost[2][1]----------------------
Best Cost[2][1] = 5
----------------------Best Cost[2][2]----------------------
Best Cost[2][2] = 6
----------------------Best Cost[3][1]----------------------
Best Cost[3][1] = 5
----------------------Best Cost[3][2]----------------------
Best Cost[2][1] = 5
Best Cost[2][2] = 6
Current Benefit : 2
Best Cost[3][2] = 7
----------------------Best Cost[3][3]----------------------
Best Cost[3][3] = 8
----------------------Best Cost[4][1]----------------------
Best Cost[4][1] = 5
----------------------Best Cost[4][2]----------------------
Best Cost[3][1] = 5
Best Cost[3][2] = 7
Current Benefit : 1
Best Cost[4][2] = 7
----------------------Best Cost[4][3]----------------------
Best Cost[3][2] = 7
Best Cost[3][3] = 8
Current Benefit : 1
Best Cost[4][3] = 8
----------------------Best Cost[4][4]----------------------
Best Cost[4][4] = 9
--------------------

In [124]:
positions = [0, 1000, 2000, 3000, 4000, 5000, 6000]

num_stations = 2

segment_burden = {
    (0, 1): 20,
    (0, 2): 55,
    (0, 3): 105,
    (0, 4): 150,
    (0, 5): 190,
    (0, 6): 230,

    (1, 2): 35,
    (1, 3): 85,
    (1, 4): 130,
    (1, 5): 170,
    (1, 6): 210,

    (2, 3): 50,
    (2, 4): 95,
    (2, 5): 135,
    (2, 6): 175,

    (3, 4): 45,
    (3, 5): 85,
    (3, 6): 125,

    (4, 5): 40,
    (4, 6): 80,

    (5, 6): 40}

In [150]:
from math import inf
from typing import Sequence


def build_dp_table(
    positions: Sequence[float],
    segment_burden: dict[tuple[int, int], float],
    num_stations: int,
) -> dict[tuple[int, int], float]:
    """
    Build the dynamic-programming table for station placement.

    DP[(i, k)] represents the minimum possible worst-segment burden
    when exactly k stations have been placed and the kth/latest
    station is at candidate point i.

    Point 0 is assumed to be the route start.
    The final point is assumed to be the route finish and is not
    considered a station candidate.

    Parameters
    ----------
    positions:
        Ordered route positions, including start and finish.

    segment_burden:
        Burden between pairs of route points, keyed by (start, end).

    num_stations:
        Exact number of stations to place.

    Returns
    -------
    dict[tuple[int, int], float]
        The completed DP table.
    """

    num_candidates = len(positions) - 2

    if num_stations < 1:
        raise ValueError("num_stations must be at least 1.")

    if num_stations > num_candidates:
        raise ValueError(
            "num_stations cannot exceed the number of candidate locations."
        )

    dp: dict[tuple[int, int], float] = {}
    parent_table : dict[tuple[int, int], int] = {}

    for total_stations in range(1, num_stations + 1):

        # The current station cannot appear before enough candidate
        # positions exist to place `total_stations` stations.
        for current_station in range(
            total_stations,
            len(positions) - 1,
        ):

            # Base case: this is the first station.
            if total_stations == 1:
                dp[(current_station, total_stations)] = (
                    segment_burden[(0, current_station)]
                )

                parent_table[(current_station, total_stations)] = 0

                continue

            best_candidate = inf

            # Try every valid location for the previous station.
            for previous_station in range(
                total_stations - 1,
                current_station,
            ):
                previous_best = dp[
                    (previous_station, total_stations - 1)
                ]

                new_segment_burden = segment_burden[
                    (previous_station, current_station)
                ]

                candidate = max(
                    previous_best,
                    new_segment_burden,
                )

                if candidate < best_candidate:
                    best_candidate = candidate

                    parent_table[
                        (current_station, total_stations)
                    ] = previous_station

            dp[(current_station, total_stations)] = best_candidate
            parent_table[(current_station, total_stations)] = 0

    return dp, parent_table

In [151]:
def find_best_final_state(
    dp: dict[tuple[int, int], float],
    segment_burden: dict[tuple[int, int], float],
    positions,
    num_stations: int,
) -> tuple[int, float]:
    """
    Find the best possible final station after accounting for
    the remaining segment from that station to the finish.

    Returns
    -------
    tuple[int, float]
        (best_last_station, best_final_score)
    """

    finish = len(positions) - 1

    best_last_station = None
    best_final_score = float("inf")

    for last_station in range(num_stations, finish):

        dp_score = dp[(last_station, num_stations)]

        burden_to_finish = segment_burden[
            (last_station, finish)
        ]

        final_score = max(
            dp_score,
            burden_to_finish,
        )

        if final_score < best_final_score:
            best_final_score = final_score
            best_last_station = last_station

    return best_last_station, best_final_score

In [155]:
dp, parent_table = build_dp_table(positions, segment_burden, num_stations)
best_last_station, best_score = find_best_final_state(
    dp,
    segment_burden,
    positions,
    num_stations=2,
)

print(best_last_station)
print(best_score)

4
95


In [156]:
parent_table

{(1, 1): 0,
 (2, 1): 0,
 (3, 1): 0,
 (4, 1): 0,
 (5, 1): 0,
 (2, 2): 1,
 (3, 2): 2,
 (4, 2): 2,
 (5, 2): 3}

In [157]:
from math import inf
from typing import Sequence


def build_dp_table(
    positions: Sequence[float],
    segment_burden: dict[tuple[int, int], float],
    num_stations: int,
) -> tuple[
    dict[tuple[int, int], float],
    dict[tuple[int, int], int],
]:
    """
    Build the dynamic-programming table for station placement.

    DP[(i, k)] represents the minimum possible worst-segment burden
    when exactly k stations have been placed and the kth/latest
    station is at candidate point i.

    parent_table[(i, k)] stores the previous station that produced
    the optimal value for DP[(i, k)].

    Point 0 is assumed to be the route start.
    The final point is assumed to be the route finish and is not
    considered a station candidate.
    """

    num_candidates = len(positions) - 2

    if num_stations < 1:
        raise ValueError("num_stations must be at least 1.")

    if num_stations > num_candidates:
        raise ValueError(
            "num_stations cannot exceed the number of candidate locations."
        )

    dp: dict[tuple[int, int], float] = {}
    parent_table: dict[tuple[int, int], int] = {}

    for total_stations in range(1, num_stations + 1):

        # The kth station cannot appear before candidate k.
        for current_station in range(
            total_stations,
            len(positions) - 1,
        ):

            # Base case: exactly one station.
            if total_stations == 1:
                dp[(current_station, total_stations)] = (
                    segment_burden[(0, current_station)]
                )

                # 0 represents the route start.
                parent_table[(current_station, total_stations)] = 0

                continue

            best_candidate = inf

            # Try every valid location for the previous station.
            for previous_station in range(
                total_stations - 1,
                current_station,
            ):
                previous_best = dp[
                    (previous_station, total_stations - 1)
                ]

                new_segment_burden = segment_burden[
                    (previous_station, current_station)
                ]

                # The score of this configuration is its worst
                # uninterrupted segment so far.
                candidate = max(
                    previous_best,
                    new_segment_burden,
                )

                # Keep the best candidate and remember where it came from.
                if candidate < best_candidate:
                    best_candidate = candidate

                    parent_table[
                        (current_station, total_stations)
                    ] = previous_station

            dp[(current_station, total_stations)] = best_candidate

    return dp, parent_table

In [158]:
def reconstruct_stations(
    parent_table: dict[tuple[int, int], int],
    best_last_station: int,
    num_stations: int,
) -> list[int]:

    stations = []

    current_station = best_last_station
    remaining_stations = num_stations

    while remaining_stations > 0:
        stations.append(current_station)

        current_station = parent_table[
            (current_station, remaining_stations)
        ]

        remaining_stations -= 1

    stations.reverse()

    return stations

In [159]:
station_indices = reconstruct_stations(
    parent_table,
    best_last_station,
    num_stations,
)

print(station_indices)

[2, 4]


In [160]:
station_positions = [
    positions[index]
    for index in station_indices
]